#### DICOM Metadata Extraction — MinIO → PostgreSQL

**What this notebook does:**
```
MinIO  →  customers/CUST_xxx/dicom/*.dcm
                    ↓
          Read DICOM metadata (no image processing)
                    ↓
          Extract: patient_id, scan_date, image_type,
                   modality, body_part, manufacturer ...
                    ↓
          PostgreSQL → dicom_metadata table
```

**Duplicate prevention:** `UNIQUE (minio_path) and customer id` 

**Auto-update:**
- New DICOM file added → picked up on next sync
- New customer added  → picked up automatically

## 1. Install & Import

In [1]:
!pip install pydicom


[notice] A new release of pip is available: 23.2.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# !pip install pydicom minio psycopg2-binary python-dotenv --quiet

In [2]:
import os
import io
import tempfile
import psycopg2
import pydicom
from minio import Minio
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
def get_pg():
    return psycopg2.connect(
        host="localhost", port=5432,
        dbname=os.getenv("POSTGRES_DB"),
        user=os.getenv("POSTGRES_USER"),
        password=os.getenv("POSTGRES_PASSWORD")
    )

def get_minio():
    return Minio(
        "localhost:9000",
        access_key=os.getenv("MINIO_ROOT_USER"),
        secret_key=os.getenv("MINIO_ROOT_PASSWORD"),
        secure=False
    )

pg    = get_pg()
minio = get_minio()
print("✅ Connected")

✅ Connected


---
## 2. Create `dicom_metadata` Table — Run Once

In [4]:
def create_table(pg):
    with pg.cursor() as cur:
        cur.execute("""
            CREATE TABLE IF NOT EXISTS dicom_metadata (
                id               SERIAL PRIMARY KEY,

                -- identity
                customer_id      TEXT,           -- from MinIO folder path
                patient_id       TEXT,           -- from DICOM tag (0010,0020)
                patient_name     TEXT,           -- from DICOM tag (0010,0010)
                scan_date        DATE,           -- from DICOM tag (0008,0020)

                -- scan info
                modality         TEXT,           -- CT | MR | XR | US
                image_type       TEXT,           -- e.g. CT Scan | X-Ray | MRI
                body_part        TEXT,           -- e.g. CHEST | HEAD | KNEE
                study_description TEXT,

                -- equipment
                manufacturer     TEXT,
                institution_name TEXT,

                -- file info
                file_size_mb     DECIMAL(8,2),
                minio_path       TEXT UNIQUE,    -- prevents duplicates
                indexed_at       TIMESTAMPTZ DEFAULT NOW()
            );
        """)
    pg.commit()
    print("✅ dicom_metadata table ready")


create_table(pg)

✅ dicom_metadata table ready


---
## 3. Read DICOM File & Extract Metadata

`pydicom` reads DICOM tags directly - no image rendering needed.
Uses `.get()` so missing tags return `None` instead of crashing.

In [5]:
# maps DICOM modality codes → readable image type names
MODALITY_MAP = {
    "CT": "CT Scan",
    "MR": "MRI",
    "CR": "X-Ray",
    "XR": "X-Ray",
    "DX": "X-Ray",
    "US": "Ultrasound",
    "PT": "PET Scan",
    "NM": "Nuclear Medicine",
}

def parse_dicom_date(raw_date):
    """Convert DICOM date string YYYYMMDD → Python date."""
    if not raw_date or len(str(raw_date)) < 8:
        return None
    try:
        from datetime import datetime
        return datetime.strptime(str(raw_date), "%Y%m%d").date()
    except Exception:
        return None


def extract_metadata(dcm, customer_id, object_path, file_size_bytes):
    """Extract key DICOM tags into a flat dictionary."""
    modality = str(dcm.get("Modality", "") or "")
    return {
        "customer_id":       customer_id,
        "patient_id":        str(dcm.get("PatientID",          "") or "") or None,
        "patient_name":      str(dcm.get("PatientName",        "") or "") or None,
        "scan_date":         parse_dicom_date(dcm.get("StudyDate")),
        "modality":          modality or None,
        "image_type":        MODALITY_MAP.get(modality, modality or None),
        "body_part":         str(dcm.get("BodyPartExamined",   "") or "") or None,
        "study_description": str(dcm.get("StudyDescription",  "") or "") or None,
        "manufacturer":      str(dcm.get("Manufacturer",       "") or "") or None,
        "institution_name":  str(dcm.get("InstitutionName",    "") or "") or None,
        "file_size_mb":      round(file_size_bytes / (1024 * 1024), 2),
        "minio_path":        object_path,
    }

---
## 4. Save Metadata to PostgreSQL
`ON CONFLICT DO NOTHING` on `minio_path` — no duplicates ever.

In [6]:
INSERT_SQL = """
    INSERT INTO dicom_metadata (
        customer_id, patient_id, patient_name, scan_date,
        modality, image_type, body_part, study_description,
        manufacturer, institution_name, file_size_mb, minio_path
    ) VALUES (
        %(customer_id)s, %(patient_id)s, %(patient_name)s, %(scan_date)s,
        %(modality)s, %(image_type)s, %(body_part)s, %(study_description)s,
        %(manufacturer)s, %(institution_name)s, %(file_size_mb)s, %(minio_path)s
    )
    ON CONFLICT (minio_path) DO NOTHING
"""

def save_metadata(pg, meta):
    """Insert one DICOM record. Returns True if new, False if duplicate."""
    with pg.cursor() as cur:
        cur.execute(INSERT_SQL, meta)
        inserted = cur.rowcount == 1
    pg.commit()
    return inserted

---
## 5. Process One DICOM File from MinIO
Downloads temporarily, reads tags, deletes temp file.

In [7]:
BUCKET = "health-data"

def process_dicom(minio, pg, object_path, customer_id, file_size):
    """Download DICOM from MinIO, extract metadata, save to PostgreSQL."""

    # Windows-safe temp file
    tmp = tempfile.NamedTemporaryFile(suffix=".dcm", delete=False)
    tmp_path = tmp.name
    tmp.close()

    try:
        minio.fget_object(BUCKET, object_path, tmp_path)
        dcm  = pydicom.dcmread(tmp_path, stop_before_pixels=True)  # metadata only — fast
        meta = extract_metadata(dcm, customer_id, object_path, file_size)
        inserted = save_metadata(pg, meta)
        status   = "✅ Saved" if inserted else "⏭️  Duplicate skipped"
        print(f"    {status}: {object_path}")
    except Exception as e:
        print(f"    ❌ Error: {e}")
    finally:
        if os.path.exists(tmp_path):
            os.unlink(tmp_path)

---
## 6. Sync All DICOM Files from MinIO

Scans all `customers/*/dicom/*.dcm` files.
- ✅ New customer added → picked up automatically
- ✅ New DICOM file added → only new files inserted
- ✅ Re-run anytime — duplicates silently skipped

In [8]:
def sync_dicom(minio, pg):
    """Scan MinIO and load all DICOM metadata into PostgreSQL."""
    objects = minio.list_objects(BUCKET, prefix="customers/", recursive=True)
    dicoms  = [
        obj for obj in objects
        if "dicom" in obj.object_name.lower()
        and obj.object_name.endswith(".dcm")
    ]

    print(f"Found {len(dicoms)} DICOM file(s)\n")

    for obj in dicoms:
        customer_id = obj.object_name.split("/")[1]
        print(f"👤 {customer_id}")
        process_dicom(minio, pg, obj.object_name, customer_id, obj.size)

    print("\n🎉 DICOM sync complete")


# ▶️ Run the sync
sync_dicom(minio, pg)

Found 10 DICOM file(s)

👤 CUST_amara_patel_03630F04
    ✅ Saved: customers/CUST_amara_patel_03630F04/dicom/CT_02_CUST_ama.dcm
👤 CUST_amara_patel_03630F04
    ✅ Saved: customers/CUST_amara_patel_03630F04/dicom/XR_01_CUST_ama.dcm
👤 CUST_carlos_rivera_5A81755B
    ✅ Saved: customers/CUST_carlos_rivera_5A81755B/dicom/MR_01_CUST_car.dcm
👤 CUST_carlos_rivera_5A81755B
    ✅ Saved: customers/CUST_carlos_rivera_5A81755B/dicom/XR_02_CUST_car.dcm
👤 CUST_fatima_alsayed_8594AA83
    ✅ Saved: customers/CUST_fatima_alsayed_8594AA83/dicom/CT_02_CUST_fat.dcm
👤 CUST_fatima_alsayed_8594AA83
    ✅ Saved: customers/CUST_fatima_alsayed_8594AA83/dicom/MR_01_CUST_fat.dcm
👤 CUST_john_whitfield_C4987FD5
    ✅ Saved: customers/CUST_john_whitfield_C4987FD5/dicom/MR_01_CUST_joh.dcm
👤 CUST_john_whitfield_C4987FD5
    ✅ Saved: customers/CUST_john_whitfield_C4987FD5/dicom/XR_02_CUST_joh.dcm
👤 CUST_mei_lin_66CBFE0F
    ✅ Saved: customers/CUST_mei_lin_66CBFE0F/dicom/MR_02_CUST_mei.dcm
👤 CUST_mei_lin_66CBFE0F
    ✅ Save

---
## 7. Verify — Query the Results

In [9]:
import pandas as pd

def query(pg, sql, label):
    df = pd.read_sql(sql, pg)
    print(f"\n📊 {label}")
    print(df.to_string(index=False))


# all records
query(pg, """
    SELECT customer_id, patient_id, patient_name,
           scan_date, modality, image_type,
           body_part, file_size_mb
    FROM dicom_metadata
    ORDER BY customer_id, scan_date
""", "All DICOM Records")


# summary per customer
query(pg, """
    SELECT customer_id,
           COUNT(*)                    AS total_scans,
           COUNT(DISTINCT modality)    AS scan_types,
           MIN(scan_date)              AS first_scan,
           MAX(scan_date)              AS latest_scan,
           ROUND(SUM(file_size_mb), 2) AS total_size_mb
    FROM dicom_metadata
    GROUP BY customer_id
    ORDER BY customer_id
""", "Scans per Customer")


# breakdown by image type
query(pg, """
    SELECT image_type,
           COUNT(*) AS count
    FROM dicom_metadata
    GROUP BY image_type
    ORDER BY count DESC
""", "Scans by Image Type")


📊 All DICOM Records
                 customer_id                   patient_id    patient_name  scan_date modality image_type body_part  file_size_mb
   CUST_amara_patel_03630F04    CUST_amara_patel_03630F04     Amara^Patel 2026-05-11       CT    CT Scan      None          0.01
   CUST_amara_patel_03630F04    CUST_amara_patel_03630F04     Amara^Patel 2026-05-11       XR      X-Ray      None          0.01
 CUST_carlos_rivera_5A81755B  CUST_carlos_rivera_5A81755B   Carlos^Rivera 2026-05-11       MR        MRI      None          0.01
 CUST_carlos_rivera_5A81755B  CUST_carlos_rivera_5A81755B   Carlos^Rivera 2026-05-11       XR      X-Ray      None          0.01
CUST_fatima_alsayed_8594AA83 CUST_fatima_alsayed_8594AA83 Fatima^Al-Sayed 2026-05-11       CT    CT Scan      None          0.01
CUST_fatima_alsayed_8594AA83 CUST_fatima_alsayed_8594AA83 Fatima^Al-Sayed 2026-05-11       MR        MRI      None          0.01
CUST_john_whitfield_C4987FD5 CUST_john_whitfield_C4987FD5  John^Whitfield 20

C:\Users\Nirasha J\AppData\Local\Temp\ipykernel_28484\3744554610.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, pg)
C:\Users\Nirasha J\AppData\Local\Temp\ipykernel_28484\3744554610.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, pg)
C:\Users\Nirasha J\AppData\Local\Temp\ipykernel_28484\3744554610.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, pg)
